# Colab Deployment

Use this notebook to load or train Dusty in Colab, run inference, and export browser-demo assets. The same setup also works on RunPod if you run it from a notebook environment.

## Setup

This cell is intentionally the same as the quickstart setup cell.

In [ ]:
# 1. Clone the repo and enter the directory
!git clone https://github.com/mkhordoo/dusty-llm.git
%cd dusty-llm

# 2. Install uv (takes ~1 second)
!pip install uv

# 3. Install the project using uv into the Colab system environment (takes ~3 seconds)
!uv pip install --system -e .

# 4. Pull the TinyStories and Dusty datasets via your Make command
!make download-datasets

## Option A: Download the Published SFT Checkpoint

Use this path when you only want inference and deployment assets, not training.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

checkpoint_path = Path(
    hf_hub_download(repo_id="khordoo/dusty-8m-sft", filename="dusty8m_sft.pt")
)
tokenizer_path = Path(
    hf_hub_download(repo_id="khordoo/dusty-8m-sft", filename="tokenizer.json")
)

print("checkpoint:", checkpoint_path)
print("tokenizer:", tokenizer_path)

## Chat from Python

This uses the same inference API as the browser and quickstart demos.

In [ ]:
import torch
from tiny_gpt.inference import Inference

device = "cuda" if torch.cuda.is_available() else "cpu"
engine = Inference(
    checkpoint_path=checkpoint_path,
    tokenizer_path=tokenizer_path,
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.8,
        top_p=0.9,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["hi dusty", "where are you?", "what do you dream about?"]:
    print("USER:", prompt)
    print("DUSTY:", chat(prompt))
    print()


## Option B: Train Before Deployment

Run these cells only if you want to produce a fresh checkpoint inside Colab or RunPod.

In [ ]:
!make dusty-tokenizer
!make dusty-pretrain-data
!make dusty-pretrain EPOCHS=1 CHECKPOINT_EVERY_STEPS=50
!make dusty-sft-data
!make dusty-sft-train EPOCHS=1 CHECKPOINT_EVERY_STEPS=50

## Export ONNX Browser Assets

This writes `docs/model.onnx` and `docs/tokenizer.json`, which are the files used by the static browser demo.

In [ ]:
!uv pip install --system -e '.[onnx]'
!make dusty-export-onnx ONNX_PROFILE=sft_dusty8m

## Inspect the Export

After export, download `docs/model.onnx`, `docs/tokenizer.json`, and the static files under `docs/` if you want to host the browser demo somewhere else.

In [ ]:
from pathlib import Path

for path in [
    Path("docs/model.onnx"),
    Path("docs/tokenizer.json"),
    Path("docs/index.html"),
]:
    if path.exists():
        print(f"OK {path}: {path.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(f"MISSING {path}")